# 11 — Iterative Closest Point (ICP) Scan Matching

**Section:** SLAM · **Mirrors MATLAB:** *2D Lidar SLAM Implementations* (scan-matching front-end)

ICP aligns two point clouds by alternating between (a) finding nearest-neighbor correspondences and (b) solving for the rigid transform $(R, t)$ that minimizes the sum-of-squared distances over those matches.


## Intuition — what's actually going on?

Two scans of the same room, taken from slightly different positions, look like two clouds of points that *almost* line up. ICP (Iterative Closest Point) is the algorithm that snaps them into perfect alignment.

The procedure is dead simple, repeated until convergence:

1. **Pair up** each point in the source cloud with its closest point in the target cloud.
2. Find the **best rigid transform** (rotation + translation) that brings each source point to its matched target.
3. Apply the transform. Repeat.

The magic is step 2: finding the optimal rotation given correspondences is *not* obvious, but it has a beautiful closed-form solution called the **Kabsch algorithm**. You build a small 3×3 (or 2×2 in 2D) cross-covariance matrix from the two centered clouds, take its SVD, and the optimal rotation is $R = V U^T$. One line of NumPy.

ICP is everywhere: lidar SLAM (matching consecutive scans), 3D reconstruction (stitching depth-camera frames), surgical robotics (registering CT scans to patient anatomy). The catch: it converges to the *closest* local minimum, so the initial alignment has to be reasonable.


## Analytical derivation

**Problem.** Given source points $\{p_i\}_{i=1}^N$ and target points $\{q_i\}_{i=1}^N$ (already paired), find rotation $R \in SO(d)$ and translation $t \in \mathbb{R}^d$ minimizing

$$E(R, t) \;=\; \sum_{i=1}^{N} \|R\,p_i + t - q_i\|^2$$

**Step 1 — solve for $t$ given $R$.** Setting $\partial E / \partial t = 0$:

$$\sum_i (R\,p_i + t - q_i) = 0 \quad\Longrightarrow\quad t = \bar q - R\,\bar p,\qquad \bar p = \tfrac{1}{N}\sum p_i,\ \bar q = \tfrac{1}{N}\sum q_i$$

So the optimal translation just aligns the centroids. Substituting back, the problem becomes: find $R$ minimizing

$$E'(R) \;=\; \sum_i \|R\,p'_i - q'_i\|^2,\qquad p'_i = p_i - \bar p,\ q'_i = q_i - \bar q$$

**Step 2 — solve for $R$ (Kabsch algorithm).** Expand the norm:

$$E'(R) \;=\; \sum_i (\|p'_i\|^2 + \|q'_i\|^2) - 2 \sum_i {q'_i}^T R\,p'_i$$

The first sum is constant in $R$; minimizing $E'$ is equivalent to *maximizing* $\sum_i {q'_i}^T R\,p'_i = \mathrm{tr}\!\left(R \sum_i p'_i {q'_i}^T\right) = \mathrm{tr}(R H)$ where

$$H \;=\; \sum_i p'_i\,{q'_i}^T \quad \in \mathbb{R}^{d \times d}\quad\text{(cross-covariance)}$$

With $H = U \Sigma V^T$ (SVD), write $W = V^T R U$. Since $R \in SO(d)$ and $U, V$ are orthogonal, $W \in O(d)$, so $|W_{ii}| \le 1$. Then $\mathrm{tr}(R H) = \mathrm{tr}(W \Sigma) = \sum_i W_{ii}\sigma_i \le \sum_i \sigma_i$, with equality iff $W = I$, i.e.

$$\boxed{\;R^\star \;=\; V\,U^T\;}$$

(This is the *direct* orthogonality argument — *not* the von Neumann trace inequality, which is a more general statement.)

**Reflection guard.** If $\det(V U^T) = -1$ the answer is an improper rotation (a reflection). Replace by

$$R^\star \;=\; V\,\mathrm{diag}(1, \ldots, 1, -1)\,U^T$$

(equivalent to flipping the sign of the last column of $V$ before forming $R$). **Why the last column?** It corresponds to the smallest singular value $\sigma_d$; flipping it costs only $2\sigma_d$ in the objective rather than $2\sigma_k > 2\sigma_d$ for any other column. This yields the *closest* proper rotation to the optimal improper one.

**Iteration.** ICP alternates (1) nearest-neighbor correspondence assignment and (2) optimal rigid transform; each iteration is guaranteed to weakly decrease $E$. Convergence is to a local minimum (initialization matters).

### Compatibility check — math ↔ code

| Step | Code |
|---|---|
| Nearest-neighbor correspondence | `d = np.linalg.norm(s[:, None] - tgt[None], axis=2); m = tgt[d.argmin(axis=1)]` |
| Centroids | `sm, mm = s.mean(0), m.mean(0)` |
| Cross-covariance $H$ | `H = (s - sm).T @ (m - mm)` |
| SVD $H = U \Sigma V^T$ | `U, _, Vt = np.linalg.svd(H)` |
| Optimal $R = V U^T$ | `R = Vt.T @ U.T` |
| Reflection guard | `if np.linalg.det(R) < 0: Vt[-1] *= -1; R = Vt.T @ U.T` |
| Translation $t = \bar q - R \bar p$ | `s = (R @ s.T).T + (mm - R @ sm)` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


def make_shape(n=120):
    t = np.linspace(0, 2 * np.pi, n)
    x = 3 * np.cos(t) + 0.5 * np.sin(3 * t)
    y = 3 * np.sin(t) + 0.5 * np.cos(2 * t)
    return np.column_stack([x, y])


src = make_shape()
true_angle = 0.4
true_R = np.array([[np.cos(true_angle), -np.sin(true_angle)],
                   [np.sin(true_angle),  np.cos(true_angle)]])
true_t = np.array([1.5, -1.0])
tgt = (true_R @ src.T).T + true_t + np.random.randn(*src.shape) * 0.08
print(f"True transform: angle={np.degrees(true_angle):.1f}°, t={true_t}")


In [ ]:
def icp(src, tgt, n_iter=40, tol=1e-6):
    s = src.copy()
    R_total = np.eye(2); t_total = np.zeros(2)
    prev_err = np.inf
    for _ in range(n_iter):
        # Nearest neighbors (brute force)
        d = np.linalg.norm(s[:, None] - tgt[None], axis=2)
        idx = d.argmin(axis=1)
        matched = tgt[idx]
        # Best-fit rotation via SVD
        s_mean, m_mean = s.mean(axis=0), matched.mean(axis=0)
        H = (s - s_mean).T @ (matched - m_mean)
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[-1] *= -1
            R = Vt.T @ U.T
        t = m_mean - R @ s_mean
        s = (R @ s.T).T + t
        R_total = R @ R_total
        t_total = R @ t_total + t
        err = np.mean(np.linalg.norm(s - matched, axis=1) ** 2)
        if abs(prev_err - err) < tol:
            break
        prev_err = err
    return R_total, t_total, s


R_est, t_est, aligned = icp(src.copy(), tgt)
print(f"Recovered: angle={np.degrees(np.arctan2(R_est[1, 0], R_est[0, 0])):.2f}°, t={t_est}")


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(src[:, 0], src[:, 1], 'b.', label='Source')
axs[0].plot(tgt[:, 0], tgt[:, 1], 'r.', label='Target')
axs[0].set_title('Before ICP'); axs[0].legend(); axs[0].set_aspect('equal'); axs[0].grid()

axs[1].plot(aligned[:, 0], aligned[:, 1], 'g.', label='Aligned source')
axs[1].plot(tgt[:, 0], tgt[:, 1], 'r.', label='Target')
axs[1].set_title('After ICP'); axs[1].legend(); axs[1].set_aspect('equal'); axs[1].grid()
plt.tight_layout()
plt.show()


## References & rigor notes

**Theorem** (Kabsch optimality, 1976). *Given paired centered point sets $\{p'_i\}, \{q'_i\} \subset \mathbb{R}^d$, the proper rotation $R \in SO(d)$ minimizing $\sum_i \|R p'_i - q'_i\|^2$ is $R^\star = V U^T$ where $H = \sum_i p'_i {q'_i}^T = U \Sigma V^T$ is the SVD of the cross-covariance. If $\det(VU^T) = -1$, flip the last column of $V$ to enforce a proper rotation.*

**Monotone descent.** Each ICP iteration weakly decreases the sum-of-squared distances (proof: the closest-point assignment step can only decrease it, and the Kabsch step minimizes over $R, t$ given the assignment). Hence the algorithm converges — to a *local* minimum.

**Complexity per iteration.** Brute-force NN: $O(N^2)$. With kd-tree: $O(N \log N)$. SVD of $d \times d$ matrix: $O(d^3)$, negligible.

**References.**
- Besl, P. J., & McKay, N. D. (1992). *A method for registration of 3-D shapes*. IEEE Trans. PAMI, 14(2), 239-256.
- Kabsch, W. (1976). *A solution for the best rotation to relate two sets of vectors*. Acta Crystallographica, A32(5), 922-923.
- Pomerleau, F., Colas, F., & Siegwart, R. (2015). *A review of point cloud registration algorithms for mobile robotics*. Foundations and Trends in Robotics, 4(1).
